In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
import sys
!{sys.executable} -m pip install seaborn



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
"c:\Program" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [3]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [4]:
import pandas as pd
import numpy as np
#import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:

import pandas as pd
import awswrangler as wr

# ===============================================================
# 1. Parámetros de tu bucket y prefijo
# ===============================================================
bucket_name  = 'ibk-discovery-comercial-us-east-1-654654352211-data'
# ⚠ Ruta corregida: incluye /INFERENCIA/ (confirmado por diagnóstico)
model_prefix = 'discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/INFERENCIA'

PERIODOS_LECTURA = [202508, 202509]

# ===============================================================
# 2. Función de lectura por periodos desde S3 (parquet sin extensión)
# ===============================================================
def LecturaDatos(meses, cols_exclude=[], boto3_session=None):
    """
    Lee archivos parquet desde S3 particionados por periodo=XXXXXX/.
    Los archivos no tienen extensión .parquet (formato Athena/Presto).
    """
    base_s3 = f"s3://{bucket_name}/{model_prefix}"
    dfs = []

    for mes in meses:
        path = f"{base_s3}/periodo={mes}/"
        print(f"  Leyendo periodo {mes}  →  {path}")
        try:
            # list_objects para verificar que existan archivos
            objs = wr.s3.list_objects(path, boto3_session=boto3_session)
            if not objs:
                print(f"    ⚠ Sin archivos en {path}")
                continue

            # dataset=True + ignore_index maneja archivos sin extensión .parquet
            df2 = wr.s3.read_parquet(
                path=path,
                boto3_session=boto3_session,
                dataset=True,
                use_threads=True,
            )
            if cols_exclude:
                df2 = df2.drop(columns=[c for c in cols_exclude if c in df2.columns])
            dfs.append(df2)
            print(f"    ✓ {df2.shape[0]:,} filas  |  {df2.shape[1]} cols")
        except Exception as e:
            print(f"    ⚠ Error en periodo {mes}: {e}")

    if not dfs:
        print("\n❌ No se cargaron datos. Verifica los periodos disponibles "
              "ejecutando la celda de diagnóstico.")
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    df.reset_index(inplace=True, drop=True)
    return df

# ===============================================================
# 3. Lectura de datos de TRAIN
# ===============================================================
df_test = LecturaDatos(PERIODOS_LECTURA, boto3_session=session)

print(f"\n✓ Shape df_train: {df_test.shape}")
if not df_test.empty:
    display(df_test.head())

  Leyendo periodo 202508  →  s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo=202508/
    ✓ 1,598,063 filas  |  103 cols
  Leyendo periodo 202509  →  s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo=202509/
    ✓ 1,573,885 filas  |  103 cols

✓ Shape df_train: (3171948, 103)


,cuc_num,cod_mes,mes,subsegmento,target,tipo_origen,key_value,num_antiguedad,flg_activo,flg_inteligo,...,fe_ratio_salidas_entradas,fe_ratio_transferencias_vs_total,fe_velocidad_rotacion,fe_zscore_abonos,fe_zscore_cargos,fe_ros_por_antiguedad,fe_estructuracion_ratio,fe_exterior_vs_pasivo,fe_pep_exposure,periodo
0,0009780075,202508,2025-08-01,Masivo,0,nunca_alertado,1D8BEBC3AB37AF43F140C5C7D4C0D8900FBA72036B8C79...,19,1,0,...,0.00,0.00,0.00,0.54,0.00,0.00,0.00,0.00,0.00,202508
1,0013462719,202508,2025-08-01,Masivo,0,nunca_alertado,B65339E15A2E094476E107F2883BE79F7C076DED9C8711...,11,0,0,...,0.54,0.47,2.28,-1.00,-1.00,0.00,0.00,0.00,0.00,202508
2,0015429829,202508,2025-08-01,Masivo,0,nunca_alertado,E38105B5B4952C4D9B1CC909E103630BADFB117C786022...,7,1,0,...,41504.00,0.00,0.35,0.00,1.00,0.00,0.00,0.00,0.00,202508
3,0009656834,202508,2025-08-01,Masivo,0,nunca_alertado,1BAFA347037388404A78C0532A9BFAE2E9F1992F931687...,19,0,0,...,1.12,0.33,4.31,-1.00,-1.00,0.00,0.00,0.00,0.00,202508
4,0013834075,202508,2025-08-01,Masivo,0,nunca_alertado,502844195CA73BC0FE37714F27E585F160436E1715F608...,10,1,0,...,0.98,0.00,60.96,-1.00,-1.00,0.00,0.00,0.00,0.00,202508


In [6]:
df_inference=df_test

In [7]:
df_inference.shape

(3171948, 103)

In [8]:
list(df_inference)

['cuc_num',
 'cod_mes',
 'mes',
 'subsegmento',
 'target',
 'tipo_origen',
 'key_value',
 'num_antiguedad',
 'flg_activo',
 'flg_inteligo',
 'cnt_dif_abn_crgsefe_6m',
 'flg_vrcn_abonos_5m_1m',
 'imp_trx_abonosefect_3m',
 'imp_trx_cargosefe_12m',
 'max_trx_cargos_12m',
 'imp_trx_cargostot_1m',
 'imp_trx_cargosefe_3m',
 'imp_trx_cargostot_3m',
 'cnt_trx_abonosefect_12m',
 'cnt_trx_abonostot_12m',
 'cnt_trx_cargostot_12m',
 'max_trx_abonos_1m',
 'imp_trx_abonostot_3m',
 'imp_trx_abonostot_9m',
 'imp_trx_abonosefect_12m',
 'cnt_meses_siningresos_12m',
 'cnt_meses_sinegresos_12m',
 'mto_pas_soles',
 'cnt_ros_hist',
 'flg_ros_12m',
 'mto_ro_debajo_umbral',
 'cnt_ro_debajo_umbral',
 'imp_trx_debajo10k_ing_12m',
 'cnt_trx_debajo10k_ing_12m',
 'imp_trx_debajo10k_egr_12m',
 'cnt_trx_debajo10k_egr_12m',
 'flg_al_ext_12m',
 'flg_del_ext_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'mto_ing_delextrsgalto_12m',
 'mto_al_extrsgaltot_12m',
 'rat_ing_dl_xt_lt_tot_12m',
 'rat_mto_al_xt_lt_12m',
 'flg_p

In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL   = "target"
PERIODO_COL  = "cod_mes"
PERIODOS     = [202508, 202509]
OUTPUT_DIR   = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\bivariados_graficos_test"
N_BINS       = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Variables seleccionadas ──────────────────────────────────────────────────
VARIABLES_SELECCIONADAS = [
  'num_antiguedad',
  'flg_activo',
  'flg_inteligo',
  'cnt_dif_abn_crgsefe_6m',
  'flg_vrcn_abonos_5m_1m',
  'imp_trx_abonosefect_3m',
  'imp_trx_cargosefe_12m',
  'max_trx_cargos_12m',
  'imp_trx_cargostot_1m',
  'imp_trx_cargosefe_3m',
  'imp_trx_cargostot_3m',
  'cnt_trx_abonosefect_12m',
  'cnt_trx_abonostot_12m',
  'cnt_trx_cargostot_12m',
  'max_trx_abonos_1m',
  'imp_trx_abonostot_3m',
  'imp_trx_abonostot_9m',
  'imp_trx_abonosefect_12m',
  'cnt_meses_siningresos_12m',
  'cnt_meses_sinegresos_12m',
  'mto_pas_soles',
  'cnt_ros_hist',
  'flg_ros_12m',
  'mto_ro_debajo_umbral',
  'cnt_ro_debajo_umbral',
  'imp_trx_debajo10k_ing_12m',
  'cnt_trx_debajo10k_ing_12m',
  'imp_trx_debajo10k_egr_12m',
  'cnt_trx_debajo10k_egr_12m',
  'flg_al_ext_12m',
  'flg_del_ext_12m',
  'mto_al_ext_12m',
  'mto_del_ext_12m',
  'mto_ing_delextrsgalto_12m',
  'mto_al_extrsgaltot_12m',
  'rat_ing_dl_xt_lt_tot_12m',
  'rat_mto_al_xt_lt_12m',
  'flg_pep',
  'cod_rsg_pep',
  'cod_v01_lista_rsg',
  'mto_cp_pep_ing',
  'mto_cp_pep_egr',
  'flg_t1cp_rosing_12m',
  'flg_cp_ros_egr_12m',
  'mto_cp_tot_ing_ros',
  'mto_cp_tot_egr_ros',
  'cod_v12_lugar_rsdn_rsg',
  'cod_v13_lugar_op_rsg_12m',
  'cod_v11_pais_op_rsg',
  'cod_v16_canal_op_rsg_12m',
  'mto_ing_tnda_rsg_alto_12m',
  'cnt_tienda_rsg_alto_12m',
  'mto_ing_cnl_rsg_alto_12m',
  'cnt_canal_rsg_alto_12m',
  'flg_alerta_12m',
  'cnt_alerta_hist',
  'flg_vrcn_efe_cargos_5m_1m',
  'mto_dif_abn_crgsefe_6m',
  'rat_trx_mntabnsefetot_6m',
  'rat_mntcrgsefetot_6m',
  'concentracion_dia_max',
  'monto_debitos_soles',
  'monto_efectivo_soles',
  'monto_max_soles',
  'monto_promedio_dolares',
  'monto_std_soles',
  'monto_total_dolares',
  'monto_transferencias_soles',
  'n_canales_distintos',
  'n_consumos',
  'n_cuentas_distintas',
  'n_trx_app',
  'n_trx_efectivo',
  'n_trx_mismo_cliente',
  'n_trx_otros',
  'n_trx_pagos',
  'ratio_debito_credito',
  'tc_total_monto_mes',
  'tc_std_monto_mes',
  'tc_pct_no_autorizadas',
  'td_avg_monto_1h',
  'td_monto_total',
  'td_gap_promedio',
  'fe_monto_total_combinado',
  'fe_pct_trx_fuera_horario',
  'fe_ratio_efectivo_vs_total',
  'fe_ratio_salidas_entradas',
  'fe_ratio_transferencias_vs_total',
  'fe_velocidad_rotacion',
  'fe_zscore_abonos',
  'fe_zscore_cargos',
  'fe_ros_por_antiguedad',
  'fe_estructuracion_ratio',
  'fe_pep_exposure',
]

# ── Validar columnas existentes ANTES de cargar ──────────────────────────────
variables     = [c for c in VARIABLES_SELECCIONADAS if c in df_inference.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df_inference.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas en el DataFrame: {no_existentes}")
print(f"✓ Variables disponibles: {len(variables)}")

# ── Filtrar: SOLO columnas necesarias + reemplazar inf → NaN ─────────────────
cols_necesarias = [PERIODO_COL, TARGET_COL] + variables
cols_necesarias = [c for c in cols_necesarias if c in df_inference.columns]

mask = df_inference[PERIODO_COL].astype(int).isin(PERIODOS)
df   = df_inference.loc[mask, cols_necesarias].copy()

# Convertir a numérico y limpiar inf/-inf
for col in variables:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df[variables] = df[variables].replace([np.inf, -np.inf], np.nan)

print(f"✓ Registros filtrados: {df.shape[0]:,}  |  Target rate: {df[TARGET_COL].mean():.4f}")
print(f"✓ Variables a graficar: {len(variables)}")

# ── Función de análisis bivariado ────────────────────────────────────────────
def plot_bivariado(df, col, target, n_bins, output_dir):
    df_tmp = df[[col, target]].copy()
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")
    df_tmp[col] = df_tmp[col].replace([np.inf, -np.inf], np.nan)
    df_tmp = df_tmp.dropna(subset=[col])

    if df_tmp.empty:
        return None

    try:
        df_tmp["bin"] = pd.qcut(df_tmp[col], q=n_bins, duplicates="drop")
    except Exception:
        df_tmp["bin"] = pd.cut(df_tmp[col], bins=n_bins)

    resumen = (
        df_tmp.groupby("bin", observed=True)[target]
        .agg(count="count", target_rate="mean")
        .reset_index()
    )
    resumen["bin_str"] = resumen["bin"].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.bar(resumen["bin_str"], resumen["count"], color="#4C72B0", alpha=0.7, label="Registros")
    ax1.set_xlabel(col, fontsize=11)
    ax1.set_ylabel("Registros", fontsize=11, color="#4C72B0")
    ax1.tick_params(axis="x", rotation=45, labelsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    ax2 = ax1.twinx()
    ax2.plot(resumen["bin_str"], resumen["target_rate"], color="#DD4949",
             marker="o", linewidth=2, label="Target rate")
    ax2.set_ylabel("Target rate", fontsize=11, color="#DD4949")
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=2))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    plt.title(f"Bivariado: {col}  |  periodos {PERIODOS[0]}–{PERIODOS[-1]}", fontsize=13)
    plt.tight_layout()

    filepath = os.path.join(output_dir, f"biv_{col}.png")
    plt.savefig(filepath, dpi=120, bbox_inches="tight")
    plt.close()
    return filepath

# ── Loop principal ────────────────────────────────────────────────────────────
guardados = []
errores   = []

for i, col in enumerate(variables, 1):
    try:
        fp = plot_bivariado(df, col, TARGET_COL, N_BINS, OUTPUT_DIR)
        if fp:
            guardados.append(fp)
            print(f"  [{i}/{len(variables)}] {col} ✓")
        else:
            errores.append((col, "sin datos válidos tras limpiar inf/NaN"))
    except Exception as e:
        errores.append((col, str(e)))

print(f"\n✅ Gráficos guardados: {len(guardados)}")
print(f"⚠  Errores:           {len(errores)}")
if errores:
    for col, err in errores:
        print(f"   → {col}: {err}")
print(f"\n📁 Carpeta: {OUTPUT_DIR}")


✓ Variables disponibles: 94
✓ Registros filtrados: 3,171,948  |  Target rate: 0.0002
✓ Variables a graficar: 94
  [1/94] num_antiguedad ✓
  [2/94] flg_activo ✓
  [3/94] flg_inteligo ✓
  [4/94] cnt_dif_abn_crgsefe_6m ✓
  [5/94] flg_vrcn_abonos_5m_1m ✓
  [6/94] imp_trx_abonosefect_3m ✓
  [7/94] imp_trx_cargosefe_12m ✓
  [8/94] max_trx_cargos_12m ✓
  [9/94] imp_trx_cargostot_1m ✓
  [10/94] imp_trx_cargosefe_3m ✓
  [11/94] imp_trx_cargostot_3m ✓
  [12/94] cnt_trx_abonosefect_12m ✓
  [13/94] cnt_trx_abonostot_12m ✓
  [14/94] cnt_trx_cargostot_12m ✓
  [15/94] max_trx_abonos_1m ✓
  [16/94] imp_trx_abonostot_3m ✓
  [17/94] imp_trx_abonostot_9m ✓
  [18/94] imp_trx_abonosefect_12m ✓
  [19/94] cnt_meses_siningresos_12m ✓
  [20/94] cnt_meses_sinegresos_12m ✓
  [21/94] mto_pas_soles ✓
  [22/94] cnt_ros_hist ✓
  [23/94] flg_ros_12m ✓
  [24/94] mto_ro_debajo_umbral ✓
  [25/94] cnt_ro_debajo_umbral ✓
  [26/94] imp_trx_debajo10k_ing_12m ✓
  [27/94] cnt_trx_debajo10k_ing_12m ✓
  [28/94] imp_trx_debajo10

In [10]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ── Parámetros ──────────────────────────────────────────────────────────────
TARGET_COL  = "target"
PERIODO_COL = "cod_mes"
PERIODOS    = [202508, 202509]
OUTPUT_DIR  = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test"
N_BINS      = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)

VARIABLES_SELECCIONADAS = [
  'num_antiguedad',
  'flg_activo',
  'flg_inteligo',
  'cnt_dif_abn_crgsefe_6m',
  'flg_vrcn_abonos_5m_1m',
  'imp_trx_abonosefect_3m',
  'imp_trx_cargosefe_12m',
  'max_trx_cargos_12m',
  'imp_trx_cargostot_1m',
  'imp_trx_cargosefe_3m',
  'imp_trx_cargostot_3m',
  'cnt_trx_abonosefect_12m',
  'cnt_trx_abonostot_12m',
  'cnt_trx_cargostot_12m',
  'max_trx_abonos_1m',
  'imp_trx_abonostot_3m',
  'imp_trx_abonostot_9m',
  'imp_trx_abonosefect_12m',
  'cnt_meses_siningresos_12m',
  'cnt_meses_sinegresos_12m',
  'mto_pas_soles',
  'cnt_ros_hist',
  'flg_ros_12m',
  'mto_ro_debajo_umbral',
  'cnt_ro_debajo_umbral',
  'imp_trx_debajo10k_ing_12m',
  'cnt_trx_debajo10k_ing_12m',
  'imp_trx_debajo10k_egr_12m',
  'cnt_trx_debajo10k_egr_12m',
  'flg_al_ext_12m',
  'flg_del_ext_12m',
  'mto_al_ext_12m',
  'mto_del_ext_12m',
  'mto_ing_delextrsgalto_12m',
  'mto_al_extrsgaltot_12m',
  'rat_ing_dl_xt_lt_tot_12m',
  'rat_mto_al_xt_lt_12m',
  'flg_pep',
  'cod_rsg_pep',
  'cod_v01_lista_rsg',
  'mto_cp_pep_ing',
  'mto_cp_pep_egr',
  'flg_t1cp_rosing_12m',
  'flg_cp_ros_egr_12m',
  'mto_cp_tot_ing_ros',
  'mto_cp_tot_egr_ros',
  'cod_v12_lugar_rsdn_rsg',
  'cod_v13_lugar_op_rsg_12m',
  'cod_v11_pais_op_rsg',
  'cod_v16_canal_op_rsg_12m',
  'mto_ing_tnda_rsg_alto_12m',
  'cnt_tienda_rsg_alto_12m',
  'mto_ing_cnl_rsg_alto_12m',
  'cnt_canal_rsg_alto_12m',
  'flg_alerta_12m',
  'cnt_alerta_hist',
  'flg_vrcn_efe_cargos_5m_1m',
  'mto_dif_abn_crgsefe_6m',
  'rat_trx_mntabnsefetot_6m',
  'rat_mntcrgsefetot_6m',
  'concentracion_dia_max',
  'monto_debitos_soles',
  'monto_efectivo_soles',
  'monto_max_soles',
  'monto_promedio_dolares',
  'monto_std_soles',
  'monto_total_dolares',
  'monto_transferencias_soles',
  'n_canales_distintos',
  'n_consumos',
  'n_cuentas_distintas',
  'n_trx_app',
  'n_trx_efectivo',
  'n_trx_mismo_cliente',
  'n_trx_otros',
  'n_trx_pagos',
  'ratio_debito_credito',
  'tc_total_monto_mes',
  'tc_std_monto_mes',
  'tc_pct_no_autorizadas',
  'td_avg_monto_1h',
  'td_monto_total',
  'td_gap_promedio',
  'fe_monto_total_combinado',
  'fe_pct_trx_fuera_horario',
  'fe_ratio_efectivo_vs_total',
  'fe_ratio_salidas_entradas',
  'fe_ratio_transferencias_vs_total',
  'fe_velocidad_rotacion',
  'fe_zscore_abonos',
  'fe_zscore_cargos',
  'fe_ros_por_antiguedad',
  'fe_estructuracion_ratio',
  'fe_pep_exposure',
]

# ── Validar columnas ANTES de cargar ────────────────────────────────────────
variables     = [c for c in VARIABLES_SELECCIONADAS if c in df_inference.columns]
no_existentes = [c for c in VARIABLES_SELECCIONADAS if c not in df_inference.columns]

if no_existentes:
    print(f"⚠ Columnas no encontradas: {no_existentes}")

# ── Filtrar: SOLO columnas necesarias (ahorra RAM) ───────────────────────────
cols_necesarias = [PERIODO_COL, TARGET_COL] + variables
cols_necesarias = [c for c in cols_necesarias if c in df_inference.columns]

mask   = df_inference[PERIODO_COL].astype(int).isin(PERIODOS)
df_eda = df_inference.loc[mask, cols_necesarias].copy()

# Convertir a numérico y reemplazar inf/-inf → NaN
for col in variables:
    df_eda[col] = pd.to_numeric(df_eda[col], errors="coerce")

# Reemplazar inf en columnas numéricas únicamente
num_cols = df_eda[variables].select_dtypes(include=[np.number]).columns.tolist()
df_eda[num_cols] = df_eda[num_cols].replace([np.inf, -np.inf], np.nan)

print(f"✓ Shape EDA: {df_eda.shape}")
print(f"✓ Target rate: {df_eda[TARGET_COL].mean():.4f}")


# ════════════════════════════════════════════════════════════════════════════
# 1. RESUMEN ESTADÍSTICO
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("1. RESUMEN ESTADÍSTICO")
print("="*60)

stats = df_eda[variables].describe().T
stats["missing"]     = df_eda[variables].isnull().sum()
stats["missing_pct"] = (df_eda[variables].isnull().mean() * 100).round(2)
stats["zeros_pct"]   = ((df_eda[variables] == 0).mean() * 100).round(2)
stats["skewness"]    = df_eda[variables].skew(numeric_only=True).round(3)
stats["kurtosis"]    = df_eda[variables].kurt(numeric_only=True).round(3)

display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max",
               "missing_pct", "zeros_pct", "skewness", "kurtosis"]])

stats.to_csv(os.path.join(OUTPUT_DIR, "01_resumen_estadistico.csv"))
print(f"✓ Guardado: 01_resumen_estadistico.csv")


# ════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("2. MISSING VALUES")
print("="*60)

missing = df_eda[variables].isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

if missing.empty:
    print("✓ No hay missing values")
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing) * 0.35)))
    missing.plot(kind="barh", ax=ax, color="#E07B54")
    ax.set_xlabel("% Missing", fontsize=11)
    ax.set_title("Missing Values por Variable (%)", fontsize=13)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    for i, v in enumerate(missing):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_missing_values.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"✓ Guardado: 02_missing_values.png")
    display(missing.to_frame("missing_pct"))


# ════════════════════════════════════════════════════════════════════════════
# 3. DISTRIBUCIONES (histogramas)
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("3. DISTRIBUCIONES")
print("="*60)

n_cols  = 4
n_rows  = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    data = pd.to_numeric(df_eda[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if data.empty:
        axes[i].set_title(f"{col}\n(sin datos)", fontsize=7, pad=3)
        axes[i].axis("off")
        continue
    axes[i].hist(data, bins=40, color="#4C72B0", alpha=0.75, edgecolor="white")
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.1f}")
    ))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuciones de Variables", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_distribuciones.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 03_distribuciones.png")


# ════════════════════════════════════════════════════════════════════════════
# 4. CORRELACIÓN CON EL TARGET
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("4. CORRELACIÓN CON EL TARGET")
print("="*60)

corr_target = (
    df_eda[variables + [TARGET_COL]]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL, errors="ignore")
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_target) * 0.35)))
colors = ["#DD4949" if v > 0 else "#4C72B0" for v in corr_target]
corr_target.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson", fontsize=11)
ax.set_title(f"Correlación de Variables con '{TARGET_COL}'", fontsize=13)
for i, v in enumerate(corr_target):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_correlacion_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 04_correlacion_target.png")

display(corr_target.to_frame("corr_con_target").style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))
corr_target.to_frame("corr_con_target").to_csv(os.path.join(OUTPUT_DIR, "04_correlacion_target.csv"))


# ════════════════════════════════════════════════════════════════════════════
# 5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES")
print("="*60)

corr_matrix = df_eda[variables].corr(numeric_only=True)
mask_tri = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_plot = corr_matrix.copy()
corr_plot[mask_tri] = np.nan

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(corr_plot, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.6)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        val = corr_plot.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=5.5, color="black" if abs(val) < 0.7 else "white")
ax.set_title("Matriz de Correlación entre Variables", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_matriz_correlacion.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 05_matriz_correlacion.png")

corr_upper = corr_matrix.where(mask_tri == False).stack().reset_index()
corr_upper.columns = ["var1", "var2", "correlacion"]
corr_upper = corr_upper[corr_upper["var1"] != corr_upper["var2"]]
alta_corr  = corr_upper[corr_upper["correlacion"].abs() > 0.7].sort_values("correlacion", key=abs, ascending=False)
print(f"\n⚠ Pares con correlación > 0.7:")
display(alta_corr)
alta_corr.to_csv(os.path.join(OUTPUT_DIR, "05_alta_correlacion.csv"), index=False)


# ════════════════════════════════════════════════════════════════════════════
# 6. BOXPLOT TARGET=0 vs TARGET=1
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("6. DISTRIBUCIÓN POR TARGET (0 vs 1)")
print("="*60)

n_cols = 4
n_rows = int(np.ceil(len(variables) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(variables):
    g0 = pd.to_numeric(df_eda[df_eda[TARGET_COL] == 0][col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    g1 = pd.to_numeric(df_eda[df_eda[TARGET_COL] == 1][col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    axes[i].boxplot([g0, g1], tick_labels=["target=0", "target=1"],
                    patch_artist=True,
                    boxprops=dict(facecolor="#4C72B0", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2))
    axes[i].set_title(col, fontsize=8, pad=3)
    axes[i].tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución por Target (0 vs 1)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_boxplot_por_target.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Guardado: 06_boxplot_por_target.png")


# ════════════════════════════════════════════════════════════════════════════
# RESUMEN FINAL
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : {len(variables):<25}║
║  Registros            : {df_eda.shape[0]:<25,}║
║  Target rate          : {df_eda[TARGET_COL].mean():<25.4f}║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")


✓ Shape EDA: (3171948, 96)
✓ Target rate: 0.0002

1. RESUMEN ESTADÍSTICO


,count,mean,std,min,25%,50%,75%,max,missing_pct,zeros_pct,skewness,kurtosis
num_antiguedad,3171948.00,11.06,21.84,0.00,4.00,9.00,15.00,935.00,0.00,6.27,33.43,1391.27
flg_activo,2588274.00,0.47,0.50,0.00,0.00,0.00,1.00,1.00,18.40,53.23,0.13,-1.98
flg_inteligo,3171948.00,0.00,0.01,0.00,0.00,0.00,0.00,1.00,0.00,100.00,197.88,39154.91
cnt_dif_abn_crgsefe_6m,3165943.00,-4.43,16.92,-539.00,-4.00,0.00,0.00,6004.00,0.19,42.32,65.74,15365.67
flg_vrcn_abonos_5m_1m,3171948.00,0.49,0.50,0.00,0.00,0.00,1.00,1.00,0.00,50.80,0.03,-2.00
...,...,...,...,...,...,...,...,...,...,...,...,...
fe_zscore_abonos,3171948.00,-0.07,0.95,-1.00,-1.00,-0.25,1.00,1.00,0.00,4.19,0.13,-1.90
fe_zscore_cargos,3171948.00,-0.04,0.95,-1.00,-1.00,0.00,1.00,1.00,0.00,9.49,0.09,-1.89
fe_ros_por_antiguedad,3171948.00,0.00,0.01,0.00,0.00,0.00,0.00,3.67,0.00,99.86,121.22,26492.01
fe_estructuracion_ratio,3171948.00,0.00,0.00,0.00,0.00,0.00,0.00,0.44,0.00,99.99,461.50,280411.57


✓ Guardado: 01_resumen_estadistico.csv

2. MISSING VALUES
✓ Guardado: 02_missing_values.png


,missing_pct
cnt_trx_debajo10k_egr_12m,100.00
imp_trx_debajo10k_egr_12m,100.00
mto_ro_debajo_umbral,99.99
imp_trx_debajo10k_ing_12m,99.99
cnt_trx_debajo10k_ing_12m,99.99
mto_cp_tot_egr_ros,99.97
mto_cp_pep_egr,99.85
mto_cp_pep_ing,99.85
mto_cp_tot_ing_ros,99.81
cod_v11_pais_op_rsg,99.63



3. DISTRIBUCIONES
✓ Guardado: 03_distribuciones.png

4. CORRELACIÓN CON EL TARGET
✓ Guardado: 04_correlacion_target.png


,corr_con_target
flg_alerta_12m,0.234063
mto_cp_tot_egr_ros,0.163173
imp_trx_abonostot_3m,0.112516
imp_trx_cargostot_3m,0.110476
imp_trx_cargosefe_3m,0.101381
mto_al_ext_12m,0.092023
imp_trx_abonostot_9m,0.088325
max_trx_cargos_12m,0.075821
imp_trx_abonosefect_3m,0.064511
imp_trx_cargosefe_12m,0.059031



5. MATRIZ DE CORRELACIÓN ENTRE VARIABLES
✓ Guardado: 05_matriz_correlacion.png

⚠ Pares con correlación > 0.7:


,var1,var2,correlacion
370,imp_trx_debajo10k_egr_12m,imp_trx_debajo10k_ing_12m,1.00
798,mto_cp_tot_ing_ros,imp_trx_debajo10k_ing_12m,-1.00
349,cnt_trx_debajo10k_ing_12m,imp_trx_debajo10k_ing_12m,1.00
391,cnt_trx_debajo10k_egr_12m,imp_trx_debajo10k_egr_12m,1.00
835,mto_cp_tot_egr_ros,imp_trx_debajo10k_ing_12m,-0.99
836,mto_cp_tot_egr_ros,cnt_trx_debajo10k_ing_12m,-0.99
3057,fe_monto_total_combinado,monto_debitos_soles,0.96
298,cnt_ro_debajo_umbral,mto_ro_debajo_umbral,0.93
115,imp_trx_abonostot_3m,imp_trx_cargostot_3m,0.93
1092,mto_ing_cnl_rsg_alto_12m,mto_cp_tot_ing_ros,0.89



6. DISTRIBUCIÓN POR TARGET (0 vs 1)
✓ Guardado: 06_boxplot_por_target.png

╔══════════════════════════════════════════════════╗
║              EDA COMPLETADO ✅                   ║
╠══════════════════════════════════════════════════╣
║  Variables analizadas : 94                       ║
║  Registros            : 3,171,948                ║
║  Target rate          : 0.0002                   ║
╠══════════════════════════════════════════════════╣
║  Archivos generados:                             ║
║  01_resumen_estadistico.csv                      ║
║  02_missing_values.png                           ║
║  03_distribuciones.png                           ║
║  04_correlacion_target.png / .csv                ║
║  05_matriz_correlacion.png                       ║
║  05_alta_correlacion.csv                         ║
║  06_boxplot_por_target.png                       ║
╚══════════════════════════════════════════════════╝
📁 c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_t

In [11]:

import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# SELECCIÓN DE VARIABLES PARA EL MODELO
# Criterios:
#   1. Missing % < MAX_MISSING_PCT  (hasta 99% de nulos permitido)
#   2. Sin multicolinealidad: |corr entre variables| <= CORR_THRESHOLD
#      → se elimina la de MENOR correlación con el target
# ═══════════════════════════════════════════════════════════════════════════

MAX_MISSING_PCT = 99.0   # elimina solo variables con 100% nulos
CORR_THRESHOLD  = 0.70   # correlación máxima permitida entre variables

print("=" * 65)
print("SELECCIÓN DE VARIABLES PARA EL MODELO")
print("=" * 65)
print(f"  Umbral missing   : < {MAX_MISSING_PCT}%  (se eliminan solo las que tienen 100% nulos)")
print(f"  Umbral corr vars : <= {CORR_THRESHOLD}")
print(f"  Variables iniciales: {len(variables)}\n")

# ── 1. Filtro por missing ────────────────────────────────────────────────────
missing_pct = df_eda[variables].isnull().mean() * 100
vars_ok_missing   = missing_pct[missing_pct < MAX_MISSING_PCT].index.tolist()
vars_drop_missing = missing_pct[missing_pct >= MAX_MISSING_PCT].index.tolist()

print(f"── Filtro 1: MISSING >= {MAX_MISSING_PCT}%  →  eliminadas {len(vars_drop_missing)}")
for v in vars_drop_missing:
    print(f"   ✗ {v:<45}  ({missing_pct[v]:.1f}% nulos)")
print(f"   Quedan: {len(vars_ok_missing)} variables\n")

# ── 2. Correlación con el target ─────────────────────────────────────────────
corr_con_target = (
    df_eda[vars_ok_missing + [TARGET_COL]]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL, errors="ignore")
    .abs()
    .fillna(0)
)

# ── 3. Filtro multicolinealidad (greedy, conserva la más predictiva) ──────────
corr_matrix = df_eda[vars_ok_missing].corr(numeric_only=True).abs()
vars_num    = corr_matrix.columns.tolist()

eliminadas_corr = []
seleccionadas   = sorted(vars_num, key=lambda v: corr_con_target.get(v, 0), reverse=True)

i = 0
while i < len(seleccionadas):
    v1 = seleccionadas[i]
    j  = i + 1
    while j < len(seleccionadas):
        v2 = seleccionadas[j]
        if corr_matrix.loc[v1, v2] > CORR_THRESHOLD:
            eliminadas_corr.append((v2, v1, round(corr_matrix.loc[v1, v2], 3)))
            seleccionadas.pop(j)
        else:
            j += 1
    i += 1

vars_no_num = [v for v in vars_ok_missing if v not in vars_num]

print(f"── Filtro 2: CORRELACIÓN > {CORR_THRESHOLD}  →  eliminadas {len(eliminadas_corr)}")
for v_elim, v_keep, corr_val in eliminadas_corr:
    print(f"   ✗ {v_elim:<45}  (corr={corr_val:.3f} con '{v_keep}')")
print(f"   Quedan: {len(seleccionadas)} variables numéricas\n")

if vars_no_num:
    print(f"── Variables categóricas/string (sin filtro corr): {vars_no_num}\n")

# ── Lista final ───────────────────────────────────────────────────────────────
VARIABLES_MODELO = seleccionadas + vars_no_num

# ── Tabla resumen con selector interactivo ────────────────────────────────────
df_selector = pd.DataFrame({
    "variable"    : VARIABLES_MODELO,
    "missing_pct" : [round(missing_pct.get(v, 0), 2) for v in VARIABLES_MODELO],
    "corr_target" : [round(corr_con_target.get(v, float("nan")), 6) for v in VARIABLES_MODELO],
    "tipo"        : ["numérica" if v in vars_num else "categórica" for v in VARIABLES_MODELO],
    "incluir"     : [True] * len(VARIABLES_MODELO),   # puedes editar aquí para excluir manualmente
}).sort_values("corr_target", ascending=False).reset_index(drop=True)

print("=" * 65)
print(f"✅ VARIABLES SELECCIONADAS: {len(VARIABLES_MODELO)}")
print("=" * 65)

display(
    df_selector.style
    .background_gradient(subset=["corr_target"], cmap="Greens")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .format({"missing_pct": "{:.1f}%", "corr_target": "{:.6f}"})
    .set_properties(**{"font-size": "11px"})
)

# ── Guardar tabla de selección ────────────────────────────────────────────────
OUTPUT_SEL = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\variables_modelo.csv"
df_selector.to_csv(OUTPUT_SEL, index=False)
print(f"\n✓ Tabla guardada en: {OUTPUT_SEL}")

# ── Aplicar columna 'incluir' para obtener lista final editable ───────────────
VARIABLES_MODELO_FINAL = df_selector.loc[df_selector["incluir"] == True, "variable"].tolist()

print(f"\n📋 Lista final ({len(VARIABLES_MODELO_FINAL)} variables):")
print("VARIABLES_MODELO_FINAL = [")
for v in VARIABLES_MODELO_FINAL:
    print(f"    '{v}',")
print("]")


SELECCIÓN DE VARIABLES PARA EL MODELO
  Umbral missing   : < 99.0%  (se eliminan solo las que tienen 100% nulos)
  Umbral corr vars : <= 0.7
  Variables iniciales: 94

── Filtro 1: MISSING >= 99.0%  →  eliminadas 11
   ✗ mto_ro_debajo_umbral                           (100.0% nulos)
   ✗ imp_trx_debajo10k_ing_12m                      (100.0% nulos)
   ✗ cnt_trx_debajo10k_ing_12m                      (100.0% nulos)
   ✗ imp_trx_debajo10k_egr_12m                      (100.0% nulos)
   ✗ cnt_trx_debajo10k_egr_12m                      (100.0% nulos)
   ✗ mto_al_ext_12m                                 (99.4% nulos)
   ✗ mto_cp_pep_ing                                 (99.8% nulos)
   ✗ mto_cp_pep_egr                                 (99.8% nulos)
   ✗ mto_cp_tot_ing_ros                             (99.8% nulos)
   ✗ mto_cp_tot_egr_ros                             (100.0% nulos)
   ✗ cod_v11_pais_op_rsg                            (99.6% nulos)
   Quedan: 83 variables

── Filtro 2: CORRELACIÓN > 

,variable,missing_pct,corr_target,tipo,incluir
0,flg_alerta_12m,0.0%,0.234063,numérica,True
1,imp_trx_abonostot_3m,0.1%,0.112516,numérica,True
2,imp_trx_cargosefe_3m,0.3%,0.101381,numérica,True
3,max_trx_cargos_12m,0.1%,0.075821,numérica,True
4,imp_trx_abonosefect_3m,0.3%,0.064511,numérica,True
5,mto_del_ext_12m,98.8%,0.053537,numérica,True
6,monto_transferencias_soles,0.0%,0.030178,numérica,True
7,mto_dif_abn_crgsefe_6m,0.2%,0.026275,numérica,True
8,monto_efectivo_soles,0.0%,0.025085,numérica,True
9,fe_monto_total_combinado,0.0%,0.024693,numérica,True



✓ Tabla guardada en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\variables_modelo.csv

📋 Lista final (73 variables):
VARIABLES_MODELO_FINAL = [
    'flg_alerta_12m',
    'imp_trx_abonostot_3m',
    'imp_trx_cargosefe_3m',
    'max_trx_cargos_12m',
    'imp_trx_abonosefect_3m',
    'mto_del_ext_12m',
    'monto_transferencias_soles',
    'mto_dif_abn_crgsefe_6m',
    'monto_efectivo_soles',
    'fe_monto_total_combinado',
    'cnt_trx_abonosefect_12m',
    'fe_ros_por_antiguedad',
    'mto_pas_soles',
    'cnt_trx_cargostot_12m',
    'monto_total_dolares',
    'cod_v13_lugar_op_rsg_12m',
    'concentracion_dia_max',
    'n_cuentas_distintas',
    'cnt_trx_abonostot_12m',
    'n_canales_distintos',
    'cnt_alerta_hist',
    'fe_pct_trx_fuera_horario',
    'flg_al_ext_12m',
    'cnt_canal_rsg_alto_12m',
    'cnt_dif_abn_crgsefe_6m',
    'rat_trx_mntabnsefetot_6m',
    'flg_ros_12m',
    'cnt_ro_debajo_umbral',
    'fe_ratio_transferencias_vs_total',
  

In [12]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
import os

# ═══════════════════════════════════════════════════════════════════════════
# SELECTOR POR IMPORTANCIA ACUMULADA (LightGBM)
# Conserva las variables que acumulan el UMBRAL_IMPORTANCIA de la importancia
# total del modelo.
# ═══════════════════════════════════════════════════════════════════════════

UMBRAL_IMPORTANCIA = 0.99   # 99% de importancia acumulada
OUTPUT_DIR_SEL     = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test"

# ── Preparar datos de entrenamiento ─────────────────────────────────────────
# Usamos VARIABLES_MODELO_FINAL (salida de la celda anterior)
# Solo columnas numéricas (LightGBM acepta NaN pero no strings sin encode)
vars_lgb = [v for v in VARIABLES_MODELO_FINAL
            if v in df_eda.select_dtypes(include=[np.number]).columns]

print(f"Variables numéricas para LightGBM : {len(vars_lgb)}")
print(f"Variables categóricas excluidas   : "
      f"{[v for v in VARIABLES_MODELO_FINAL if v not in vars_lgb]}\n")

X = df_eda[vars_lgb].copy()
y = df_eda[TARGET_COL].copy()

# Eliminar filas donde el target es NaN
mask_valid = y.notna()
X, y = X[mask_valid], y[mask_valid]

print(f"✓ Shape para entrenamiento: {X.shape}  |  Target rate: {y.mean():.5f}")

# ── Entrenar LightGBM rápido (solo para importancia) ────────────────────────
scale_pos = (y == 0).sum() / max((y == 1).sum(), 1)

params = {
    "objective"        : "binary",
    "metric"           : "auc",
    "n_estimators"     : 300,
    "learning_rate"    : 0.05,
    "num_leaves"       : 63,
    "min_child_samples": 50,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "scale_pos_weight" : scale_pos,
    "n_jobs"           : -1,
    "random_state"     : 42,
    "verbose"          : -1,
}

print("\n⏳ Entrenando LightGBM para calcular importancia de variables...")
model = lgb.LGBMClassifier(**params)
model.fit(X, y)
print("✓ Entrenamiento completado")

# ── Importancia acumulada ───────────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=vars_lgb)
importances = importances.sort_values(ascending=False)
importancias_norm = importances / importances.sum()
importancia_acum  = importancias_norm.cumsum()

# Variables que acumulan hasta UMBRAL_IMPORTANCIA
vars_99 = importancia_acum[importancia_acum <= UMBRAL_IMPORTANCIA].index.tolist()

# Asegurarse de incluir la variable que cruza el umbral (si quedaron justo por debajo)
if len(vars_99) < len(vars_lgb):
    siguiente = importancia_acum.index[len(vars_99)]
    vars_99.append(siguiente)

print(f"\n✅ Variables que acumulan {UMBRAL_IMPORTANCIA*100:.0f}% de importancia: {len(vars_99)} / {len(vars_lgb)}")

# ── Tabla resultado ─────────────────────────────────────────────────────────
df_importancia = pd.DataFrame({
    "variable"           : importancias_norm.index,
    "importancia"        : importancias_norm.values,
    "importancia_acum"   : importancia_acum.values,
    "missing_pct"        : [round(missing_pct.get(v, 0), 2) for v in importancias_norm.index],
    "seleccionada"       : [v in vars_99 for v in importancias_norm.index],
}).reset_index(drop=True)

display(
    df_importancia.style
    .background_gradient(subset=["importancia"], cmap="Greens")
    .background_gradient(subset=["importancia_acum"], cmap="Blues")
    .background_gradient(subset=["missing_pct"], cmap="Reds")
    .apply(lambda col: ["background-color: #d4edda" if v else "" for v in df_importancia["seleccionada"]], axis=0)
    .format({
        "importancia"      : "{:.4%}",
        "importancia_acum" : "{:.2%}",
        "missing_pct"      : "{:.1f}%",
    })
    .set_properties(**{"font-size": "11px"})
)

# ── Gráfico importancia acumulada ───────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(max(14, len(vars_lgb) * 0.25), 6))

bars = ax1.bar(range(len(importancias_norm)), importancias_norm.values,
               color=["#2ecc71" if v in vars_99 else "#bdc3c7" for v in importancias_norm.index],
               alpha=0.85, label="Importancia individual")
ax1.set_xticks(range(len(importancias_norm)))
ax1.set_xticklabels(importancias_norm.index, rotation=90, fontsize=7)
ax1.set_ylabel("Importancia relativa", fontsize=11)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2 = ax1.twinx()
ax2.plot(range(len(importancia_acum)), importancia_acum.values,
         color="#e74c3c", linewidth=2, marker=".", markersize=4, label="Importancia acumulada")
ax2.axhline(UMBRAL_IMPORTANCIA, color="#e74c3c", linestyle="--", linewidth=1,
            label=f"Umbral {UMBRAL_IMPORTANCIA*100:.0f}%")
ax2.set_ylabel("Importancia acumulada", fontsize=11, color="#e74c3c")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right", fontsize=9)

plt.title(f"Importancia de Variables (LightGBM)  |  {len(vars_99)} vars explican {UMBRAL_IMPORTANCIA*100:.0f}% de importancia",
          fontsize=13)
plt.tight_layout()
fp_imp = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.png")
plt.savefig(fp_imp, dpi=120, bbox_inches="tight")
plt.close()
print(f"✓ Gráfico guardado: {fp_imp}")

# ── Guardar tabla ───────────────────────────────────────────────────────────
fp_csv = os.path.join(OUTPUT_DIR_SEL, "07_importancia_variables.csv")
df_importancia.to_csv(fp_csv, index=False)
print(f"✓ Tabla guardada : {fp_csv}")

# ── Lista final para el modelo ──────────────────────────────────────────────
VARIABLES_FINALES_MODELO = vars_99

print(f"\n{'='*65}")
print(f"📋 VARIABLES_FINALES_MODELO ({len(VARIABLES_FINALES_MODELO)} variables)")
print(f"{'='*65}")
print("VARIABLES_FINALES_MODELO = [")
for v in VARIABLES_FINALES_MODELO:
    imp = importancias_norm.get(v, 0)
    print(f"    '{v}',  # importancia={imp:.4%}")
print("]")


Variables numéricas para LightGBM : 73
Variables categóricas excluidas   : []

✓ Shape para entrenamiento: (3171948, 73)  |  Target rate: 0.00015

⏳ Entrenando LightGBM para calcular importancia de variables...
✓ Entrenamiento completado

✅ Variables que acumulan 99% de importancia: 53 / 73


,variable,importancia,importancia_acum,missing_pct,seleccionada
0,fe_ratio_salidas_entradas,8.6008%,8.60%,0.0%,True
1,max_trx_cargos_12m,6.7838%,15.38%,0.1%,True
2,imp_trx_abonostot_3m,6.1175%,21.50%,0.1%,True
3,mto_pas_soles,5.9964%,27.50%,0.0%,True
4,monto_transferencias_soles,4.2399%,31.74%,0.0%,True
5,fe_monto_total_combinado,3.9370%,35.68%,0.0%,True
6,cnt_alerta_hist,3.6342%,39.31%,98.8%,True
7,concentracion_dia_max,3.5130%,42.82%,0.0%,True
8,cnt_trx_abonostot_12m,3.5130%,46.34%,0.1%,True
9,num_antiguedad,2.6651%,49.00%,0.0%,True


✓ Gráfico guardado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\07_importancia_variables.png
✓ Tabla guardada : c:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\07_importancia_variables.csv

📋 VARIABLES_FINALES_MODELO (53 variables)
VARIABLES_FINALES_MODELO = [
    'fe_ratio_salidas_entradas',  # importancia=8.6008%
    'max_trx_cargos_12m',  # importancia=6.7838%
    'imp_trx_abonostot_3m',  # importancia=6.1175%
    'mto_pas_soles',  # importancia=5.9964%
    'monto_transferencias_soles',  # importancia=4.2399%
    'fe_monto_total_combinado',  # importancia=3.9370%
    'cnt_alerta_hist',  # importancia=3.6342%
    'concentracion_dia_max',  # importancia=3.5130%
    'cnt_trx_abonostot_12m',  # importancia=3.5130%
    'num_antiguedad',  # importancia=2.6651%
    'rat_trx_mntabnsefetot_6m',  # importancia=2.6651%
    'imp_trx_abonosefect_3m',  # importancia=2.5439%
    'fe_velocidad_rotacion',  # importancia=2.4228%
    'm

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
# ── Rutas ────────────────────────────────────────────────────────────────────
TRAIN_CORR = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_TRAIN\04_correlacion_target.csv"
TEST_CORR  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\04_correlacion_target.csv"
TRAIN_STAT = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_TRAIN\01_resumen_estadistico.csv"
TEST_STAT  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\01_resumen_estadistico.csv"
TRAIN_ALTA = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_TRAIN\05_alta_correlacion.csv"
TEST_ALTA  = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\eda_graficos_test\05_alta_correlacion.csv"
OUTPUT_DIR = r"C:\Users\b46637\OneDrive - Interbank\PLAFT\Masivo\Desarrollo\auditoria_tier4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# A. COMPARACIÓN CORRELACIONES TRAIN vs TEST
# ════════════════════════════════════════════════════════════════════════════
df_corr_train = pd.read_csv(TRAIN_CORR, index_col=0).rename(columns={"corr_con_target": "corr_train"})
df_corr_test  = pd.read_csv(TEST_CORR,  index_col=0).rename(columns={"corr_con_target": "corr_test"})

df_corr = df_corr_train.join(df_corr_test, how="outer").reset_index()
df_corr.columns = ["variable", "corr_train", "corr_test"]
df_corr["diferencia_abs"]  = (df_corr["corr_train"] - df_corr["corr_test"]).abs()
df_corr["cambio_signo"]    = (np.sign(df_corr["corr_train"]) != np.sign(df_corr["corr_test"]))
df_corr = df_corr.sort_values("diferencia_abs", ascending=False)

print("="*60)
print("A. COMPARACIÓN CORRELACIONES TRAIN vs TEST")
print("="*60)
display(df_corr.style
    .background_gradient(subset=["diferencia_abs"], cmap="Reds")
    .applymap(lambda v: "background-color: #FFCCCC" if v else "", subset=["cambio_signo"])
    .format({"corr_train": "{:.4f}", "corr_test": "{:.4f}", "diferencia_abs": "{:.4f}"})
)

# Variables con cambio de signo (MUY sospechoso para auditoría)
cambios = df_corr[df_corr["cambio_signo"] == True]
if not cambios.empty:
    print(f"\n🔴 ALERTA: Variables con CAMBIO DE SIGNO entre Train y Test:")
    display(cambios)
else:
    print(f"\n✅ No hay cambios de signo entre Train y Test")

# Variables con diferencia > 0.02
grandes_diffs = df_corr[df_corr["diferencia_abs"] > 0.02]
if not grandes_diffs.empty:
    print(f"\n⚠️  Variables con diferencia > 0.02 entre Train y Test:")
    display(grandes_diffs)

df_corr.to_csv(os.path.join(OUTPUT_DIR, "A_correlaciones_train_vs_test.csv"), index=False)

# ════════════════════════════════════════════════════════════════════════════
# B. GRÁFICO COMPARATIVO CORRELACIONES
# ════════════════════════════════════════════════════════════════════════════
df_plot = df_corr.set_index("variable")[["corr_train", "corr_test"]].sort_values("corr_train", key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(12, max(8, len(df_plot) * 0.35)))
x = np.arange(len(df_plot))
width = 0.35
ax.barh(x - width/2, df_plot["corr_train"], width, label="Train", color="#4C72B0", alpha=0.8)
ax.barh(x + width/2, df_plot["corr_test"],  width, label="Test",  color="#DD4949", alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(df_plot.index, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación con Target", fontsize=11)
ax.set_title("Correlación con Target: Train vs Test", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "B_correlacion_train_vs_test.png"), dpi=120, bbox_inches="tight")
plt.close()
print(f"\n✓ Guardado: B_correlacion_train_vs_test.png")

# ════════════════════════════════════════════════════════════════════════════
# C. COMPARACIÓN ESTADÍSTICOS DESCRIPTIVOS TRAIN vs TEST
# ════════════════════════════════════════════════════════════════════════════
df_stat_train = pd.read_csv(TRAIN_STAT, index_col=0)
df_stat_test  = pd.read_csv(TEST_STAT,  index_col=0)

# Comparar medias y stds
comparacion_stats = pd.DataFrame({
    "mean_train":     df_stat_train["mean"],
    "mean_test":      df_stat_test["mean"],
    "std_train":      df_stat_train["std"],
    "std_test":       df_stat_test["std"],
    "missing_train":  df_stat_train["missing_pct"],
    "missing_test":   df_stat_test["missing_pct"],
    "zeros_train":    df_stat_train["zeros_pct"],
    "zeros_test":     df_stat_test["zeros_pct"],
})

comparacion_stats["drift_mean_%"] = (
    ((comparacion_stats["mean_test"] - comparacion_stats["mean_train"]) / 
     (comparacion_stats["mean_train"].abs() + 1e-10)) * 100
).round(2)

comparacion_stats["drift_std_%"] = (
    ((comparacion_stats["std_test"] - comparacion_stats["std_train"]) / 
     (comparacion_stats["std_train"].abs() + 1e-10)) * 100
).round(2)

comparacion_stats["diff_missing"] = (comparacion_stats["missing_test"] - comparacion_stats["missing_train"]).round(2)
comparacion_stats["diff_zeros"]   = (comparacion_stats["zeros_test"]   - comparacion_stats["zeros_train"]).round(2)

comparacion_stats = comparacion_stats.sort_values("drift_mean_%", key=abs, ascending=False)

print("\n" + "="*60)
print("C. DRIFT DE ESTADÍSTICOS TRAIN vs TEST")
print("="*60)
display(comparacion_stats.style
    .background_gradient(subset=["drift_mean_%", "drift_std_%"], cmap="RdYlGn_r")
    .format("{:.2f}")
)

# Alertas de drift
drift_alto = comparacion_stats[comparacion_stats["drift_mean_%"].abs() > 50]
if not drift_alto.empty:
    print(f"\n🔴 Variables con drift de media > 50% entre Train y Test:")
    display(drift_alto[["mean_train", "mean_test", "drift_mean_%"]])

comparacion_stats.to_csv(os.path.join(OUTPUT_DIR, "C_drift_estadisticos_train_vs_test.csv"))
print(f"\n✓ Guardado: C_drift_estadisticos_train_vs_test.csv")

# ════════════════════════════════════════════════════════════════════════════
# D. ALTA CORRELACIÓN ENTRE VARIABLES (multicolinealidad)
# ════════════════════════════════════════════════════════════════════════════
df_alta_train = pd.read_csv(TRAIN_ALTA)
df_alta_test  = pd.read_csv(TEST_ALTA)

print("\n" + "="*60)
print("D. MULTICOLINEALIDAD - ALTA CORRELACIÓN ENTRE VARIABLES")
print("="*60)
print(f"\n🔵 Train — pares con correlación > 0.7: {len(df_alta_train)}")
display(df_alta_train)

print(f"\n🔴 Test — pares con correlación > 0.7: {len(df_alta_test)}")
display(df_alta_test)

# Pares que aparecen en train pero no en test (o viceversa)
pares_train = set(zip(df_alta_train["var1"], df_alta_train["var2"]))
pares_test  = set(zip(df_alta_test["var1"],  df_alta_test["var2"]))
solo_train  = pares_train - pares_test
solo_test   = pares_test  - pares_train

if solo_train:
    print(f"\n⚠️  Pares con alta corr SOLO en Train (no en Test):")
    for p in solo_train: print(f"   → {p[0]}  ↔  {p[1]}")
if solo_test:
    print(f"\n⚠️  Pares con alta corr SOLO en Test (no en Train):")
    for p in solo_test: print(f"   → {p[0]}  ↔  {p[1]}")

# ════════════════════════════════════════════════════════════════════════════
# RESUMEN PARA AUDITORÍA
# ════════════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════════════╗
║         RESUMEN PARA AUDITORÍA TIER 4  ✅               ║
╠══════════════════════════════════════════════════════════╣
║  Archivos generados en:                                  ║
║  A_correlaciones_train_vs_test.csv                       ║
║  B_correlacion_train_vs_test.png                         ║
║  C_drift_estadisticos_train_vs_test.csv                  ║
╚══════════════════════════════════════════════════════════╝
📁 {OUTPUT_DIR}
""")

A. COMPARACIÓN CORRELACIONES TRAIN vs TEST


C:\Users\b46637\AppData\Local\Temp\ipykernel_21632\3249415115.py:32: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda v: "background-color: #FFCCCC" if v else "", subset=["cambio_signo"])


,variable,corr_train,corr_test,diferencia_abs,cambio_signo
65,mto_cp_tot_egr_ros,0.0230,0.1632,0.1401,False
66,mto_cp_tot_ing_ros,0.1358,0.0127,0.1231,False
67,mto_del_ext_12m,0.1473,0.0535,0.0937,False
6,cnt_ros_hist,0.0740,-0.0079,0.0819,True
33,flg_alerta_12m,0.3102,0.2341,0.0761,False
27,fe_ros_por_antiguedad,0.0824,0.0163,0.0661,False
63,mto_cp_pep_egr,0.0027,0.0524,0.0498,False
54,monto_debitos_soles,0.0676,0.0241,0.0435,False
38,flg_ros_12m,0.0494,0.0072,0.0422,False
60,monto_transferencias_soles,0.0709,0.0302,0.0407,False



🔴 ALERTA: Variables con CAMBIO DE SIGNO entre Train y Test:


,variable,corr_train,corr_test,diferencia_abs,cambio_signo
6,cnt_ros_hist,0.07,-0.01,0.08,True
0,cnt_alerta_hist,0.02,-0.01,0.03,True
74,n_canales_distintos,0.01,-0.01,0.02,True
76,n_cuentas_distintas,0.01,-0.01,0.02,True
91,td_avg_monto_1h,0.01,-0.00,0.01,True
26,fe_ratio_transferencias_vs_total,0.00,-0.01,0.01,True
22,fe_pct_trx_fuera_horario,0.00,-0.01,0.01,True
18,cod_v16_canal_op_rsg_12m,0.00,-0.00,0.01,True
24,fe_ratio_efectivo_vs_total,0.01,-0.00,0.01,True
87,ratio_debito_credito,0.00,-0.00,0.01,True


...


✓ Guardado: C_drift_estadisticos_train_vs_test.csv

D. MULTICOLINEALIDAD - ALTA CORRELACIÓN ENTRE VARIABLES

🔵 Train — pares con correlación > 0.7: 36


,var1,var2,correlacion
0,mto_cp_tot_egr_ros,mto_ro_debajo_umbral,1.00
1,mto_cp_pep_egr,cnt_trx_debajo10k_ing_12m,1.00
2,mto_cp_pep_egr,imp_trx_debajo10k_ing_12m,1.00
3,mto_cp_tot_ing_ros,imp_trx_debajo10k_ing_12m,-1.00
4,cnt_trx_debajo10k_ing_12m,imp_trx_debajo10k_ing_12m,1.00
5,mto_cp_pep_ing,mto_ro_debajo_umbral,1.00
6,cnt_trx_debajo10k_egr_12m,imp_trx_debajo10k_egr_12m,1.00
7,fe_monto_total_combinado,monto_debitos_soles,0.97
8,cnt_ro_debajo_umbral,mto_ro_debajo_umbral,0.94
9,imp_trx_abonostot_3m,imp_trx_cargostot_3m,0.93



🔴 Test — pares con correlación > 0.7: 32


,var1,var2,correlacion
0,imp_trx_debajo10k_egr_12m,imp_trx_debajo10k_ing_12m,1.00
1,mto_cp_tot_ing_ros,imp_trx_debajo10k_ing_12m,-1.00
2,cnt_trx_debajo10k_ing_12m,imp_trx_debajo10k_ing_12m,1.00
3,cnt_trx_debajo10k_egr_12m,imp_trx_debajo10k_egr_12m,1.00
4,mto_cp_tot_egr_ros,imp_trx_debajo10k_ing_12m,-0.99
5,mto_cp_tot_egr_ros,cnt_trx_debajo10k_ing_12m,-0.99
6,fe_monto_total_combinado,monto_debitos_soles,0.96
7,cnt_ro_debajo_umbral,mto_ro_debajo_umbral,0.93
8,imp_trx_abonostot_3m,imp_trx_cargostot_3m,0.93
9,mto_ing_cnl_rsg_alto_12m,mto_cp_tot_ing_ros,0.89



⚠️  Pares con alta corr SOLO en Train (no en Test):
   → fe_velocidad_rotacion  ↔  imp_trx_debajo10k_egr_12m
   → td_gap_promedio  ↔  cnt_trx_debajo10k_egr_12m
   → mto_cp_pep_egr  ↔  cnt_trx_debajo10k_ing_12m
   → monto_debitos_soles  ↔  imp_trx_cargostot_1m
   → n_cuentas_distintas  ↔  imp_trx_debajo10k_egr_12m
   → mto_cp_tot_egr_ros  ↔  mto_ro_debajo_umbral
   → flg_al_ext_12m  ↔  imp_trx_debajo10k_egr_12m
   → mto_cp_pep_egr  ↔  imp_trx_debajo10k_ing_12m
   → imp_trx_abonostot_3m  ↔  max_trx_cargos_12m
   → monto_promedio_dolares  ↔  mto_cp_tot_ing_ros
   → n_cuentas_distintas  ↔  cnt_trx_debajo10k_egr_12m
   → fe_velocidad_rotacion  ↔  cnt_trx_debajo10k_egr_12m
   → mto_cp_tot_egr_ros  ↔  mto_cp_tot_ing_ros

⚠️  Pares con alta corr SOLO en Test (no en Train):
   → monto_max_soles  ↔  monto_debitos_soles
   → imp_trx_debajo10k_egr_12m  ↔  imp_trx_debajo10k_ing_12m
   → fe_monto_total_combinado  ↔  monto_max_soles
   → cnt_trx_debajo10k_ing_12m  ↔  mto_ro_debajo_umbral
   → mto_cp